# Olfaction-Vision-Language-Embeddings

This is a quick start on loading the olfaction-vision-language models and getting the joint multimodal embeddings from an olfaction-vision data sample.

### Install Libraries

In [ ]:
!pip install transformers
!pip install safetensors
!pip install torch
!pip install torchvision

### Import and Configure

In [ ]:
import torch
from safetensors.torch import load_file
import torch.nn as nn
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
from PIL import Image


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
EMBED_DIM = 512     # Embedding dims = 512 for small, 2048 for large
ENCODER_FILE_PATH = "OVLM-Embeddings/model/ovle-small/base/olf_encoder.safetensors"
GNN_FILE_PATH = "OVLM-Embeddings/model/ovle-small/base/gnn.safetensors"

### Embeddings Function

In [ ]:
def get_embeddings(clip_model, olf_encoder, graph_model, image, olf_vec):
    """
    Gets joint olfaction-vision-language embeddings for a given image and olfaction vector.

    :param clip_model: vision-language model
    :param olf_encoder: olfactory encoder from aromas/molecules
    :param graph_model: cross-modal associator
    :param image: PIL image
    :param olf_vec: olfaction vector
    :return: joint olfaction-vision-languageembeddings
    """
    clip_model.eval()
    olf_encoder.eval()
    graph_model.eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    image_tensor = transform(image).unsqueeze(0).to(DEVICE)
    olf_tensor = torch.tensor(olf_vec, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        vision_embeds = clip_model.get_image_features(pixel_values=image_tensor)
        if EMBED_DIM != 768 and EMBED_DIM != 512:
            projection = nn.Linear(vision_embeds.shape[-1], EMBED_DIM).to(DEVICE)
            vision_embeds = projection(vision_embeds).to(DEVICE)
        vision_embeds = vision_embeds.to(DEVICE)
        olf_embeds = olf_encoder(olf_tensor).to(DEVICE)
        logits = graph_model(vision_embeds, olf_embeds).squeeze()
        prediction = torch.sigmoid(logits[0])

    return prediction.cpu().numpy()

### Get Joint Embeddings from a Data Sample

In [ ]:
# Load the models
olf_encoder = load_file(ENCODER_FILE_PATH)
graph_model = load_file(GNN_FILE_PATH)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)

# Build example vision-olfaction sample with dummy data
example_image = Image.new('RGB', (224, 224))
example_image.save(f"/tmp/image_example.jpg")
example_olf_vec = torch.randn(112)

# Run inference
embeddings = run_inference(
    clip_model,
    olf_encoder,
    graph_model,
    example_image,
    example_olf_vec
)
print("Embeddings", embeddings)